<a href="https://colab.research.google.com/github/thikhamporn0589-bit/Lab4-Geographic-Modeling-/blob/main/Lab_4_Geographic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install earthengine-api geemap

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-thikhamporn0589')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.2 MB/s eta 0:00:00


In [11]:
# =========================
# 🌍 1. DEFINE ROI
# =========================
Map = geemap.Map()

thailand = ee.FeatureCollection("FAO/GAUL/2015/level1")
roi = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Bangkok'))

Map.centerObject(roi, 10)
Map.addLayer(roi, {"color":"red"}, "Bangkok")


# =========================
# ☁️ 2. CLOUD MASK (FIX SCL)
# =========================
def maskS2(image):
    scl = image.select('SCL')

    mask = scl.neq(3) \
        .And(scl.neq(8)) \
        .And(scl.neq(9)) \
        .And(scl.neq(10)) \
        .And(scl.neq(11))

    return image.updateMask(mask)


# =========================
# 🛰️ 3. SENTINEL-2
# =========================
s2 = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterDate('2020-01-01', '2020-12-31') \
    .filterBounds(roi) \
    .map(maskS2) \
    .select(['B4', 'B8', 'B11']) \
    .median() \
    .clip(roi)

nir = s2.select('B8')
red = s2.select('B4')
swir = s2.select('B11')

# NDVI
ndvi = nir.subtract(red).divide(nir.add(red))

# NDBI
ndbi = swir.subtract(nir).divide(swir.add(nir))



In [13]:
lst = ee.ImageCollection("MODIS/061/MOD11A2") \
    .filterDate('2020-01-01', '2020-12-31') \
    .select('LST_Day_1km') \
    .mean() \
    .multiply(0.02) \
    .subtract(273.15) \
    .clip(roi)

In [14]:
dem = ee.Image("USGS/SRTMGL1_003").clip(roi)

In [16]:
def normalize(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi.geometry(),
        scale=1000,
        bestEffort=True,
        maxPixels=1e9
    )

    min_val = ee.Number(stats.values().get(0))
    max_val = ee.Number(stats.values().get(1))

    return img.subtract(min_val).divide(max_val.subtract(min_val))

In [18]:
lst_n = normalize(lst)

# NDVI → invert (cooling)
ndvi_n = ee.Image(1).subtract(normalize(ndvi))

# Built-up
ndbi_n = normalize(ndbi)

# DEM → invert
dem_n = ee.Image(1).subtract(normalize(dem))

In [19]:
model = (
    ndvi_n.multiply(0.30)
    .add(lst_n.multiply(0.25))
    .add(ndbi_n.multiply(0.25))
    .add(dem_n.multiply(0.20))
).clip(roi)

In [20]:
vis = {
    "min": 0,
    "max": 1,
    "palette": ["blue", "cyan", "yellow", "red"]
}

Map.addLayer(model, vis, "Urban Heat Risk")

In [21]:
classified = model.expression(
    "(b < 0.3) ? 1" +
    ": (b < 0.6) ? 2" +
    ": 3",
    {"b": model}
)

Map.addLayer(classified, {
    "min": 1,
    "max": 3,
    "palette": ["#2c7bb6", "#ffffbf", "#d7191c"]
}, "Heat Classes")

In [22]:
model_high = (
    ndvi_n.multiply(0.36)
    .add(lst_n.multiply(0.22))
    .add(ndbi_n.multiply(0.22))
    .add(dem_n.multiply(0.20))
).clip(roi)

model_low = (
    ndvi_n.multiply(0.24)
    .add(lst_n.multiply(0.28))
    .add(ndbi_n.multiply(0.28))
    .add(dem_n.multiply(0.20))
).clip(roi)

Map.addLayer(model_high, vis, "NDVI +20%")
Map.addLayer(model_low, vis, "NDVI -20%")

In [23]:
uncertainty = model_high.subtract(model).abs() \
    .add(model_low.subtract(model).abs())

Map.addLayer(uncertainty, {
    "min": 0,
    "max": 0.3,
    "palette": ["green", "yellow", "red"]
}, "Uncertainty")


In [24]:
samples = model.addBands(lst_n.rename('lst_norm')).sample(
    region=roi,
    scale=1000,
    numPixels=3000,
    geometries=False
)

corr = samples.reduceColumns(
    reducer=ee.Reducer.pearsonsCorrelation(),
    selectors=['constant', 'lst_norm']
)

print("Correlation (Model vs LST):", corr.getInfo())

Correlation (Model vs LST): {'correlation': 0.7947992224205066, 'p-value': 0}


In [25]:
Map

Map(center=[13.771324157733092, 100.62705713902751], controls=(WidgetControl(options=['position', 'transparent…

In [26]:
stats = model.reduceRegion(
    reducer=ee.Reducer.minMax()
        .combine(ee.Reducer.mean(), '', True)
        .combine(ee.Reducer.stdDev(), '', True),
    geometry=roi,
    scale=1000,
    maxPixels=1e9
)

print("=== Model Statistics ===")
print(stats.getInfo())

=== Model Statistics ===
{'constant_max': 0.7085503946968753, 'constant_mean': 0.4440431837695993, 'constant_min': 0.24859933108681184, 'constant_stdDev': 0.09108031365465596}


In [27]:
area = ee.Image.pixelArea().addBands(classified)

area_stats = area.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName='class'
    ),
    geometry=roi,
    scale=1000,
    maxPixels=1e9
)

print("=== Area by Class (m²) ===")
print(area_stats.getInfo())

=== Area by Class (m²) ===
{'groups': [{'class': 1, 'sum': 53367675.55661765}, {'class': 2, 'sum': 1437131101.187746}, {'class': 3, 'sum': 74806302.21495101}]}


In [29]:
stable = uncertainty.lt(0.05)

Map.addLayer(stable, {"palette":["green"]}, "Stable Area")

stable_area = ee.Image.pixelArea().updateMask(stable)

stable_stats = stable_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=1000,
    maxPixels=1e9
)

print("=== Stable Area (m²) ===")
print(stable_stats.getInfo())

=== Stable Area (m²) ===
{'area': 1283796704.3813736}


In [30]:
mean_values = model.addBands(lst_n.rename('lst')).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=1000,
    maxPixels=1e9
)

print("=== Mean Comparison ===")
print(mean_values.getInfo())

=== Mean Comparison ===
{'constant': 0.4440431837695993, 'lst': 0.28760687388088835}


In [31]:
hist = model.reduceRegion(
    reducer=ee.Reducer.histogram(),
    geometry=roi,
    scale=1000,
    maxPixels=1e9
)

print("=== Histogram ===")
print(hist.getInfo())

=== Histogram ===
{'constant': {'bucketMeans': [0.24859933108681184, 0.2501003235925483, 0.2529296875, 0.2545936424587782, 0.2568359375, 0.2587890625, 0.2607421875, 0.26280585269129664, 0.26513549109132745, 0.26658379477291994, 0.26853581942477966, 0.2705412615830637, 0.27260602332276557, 0.27358050532915723, 0.2757570137415539, 0.2788636542922145, 0.2793429571876072, 0.2828400472871412, 0.2835972987385306, 0.2859991834057919, 0.28866157705604073, 0.2906538037367985, 0.2920187732613984, 0.29362833626575446, 0.2958984375, 0.2979844023505775, 0.2997209687976091, 0.30233596504263627, 0.3039892896382684, 0.30565082225950707, 0.30729485607541646, 0.30940491419520416, 0.31165988831650576, 0.31314034355193304, 0.3153853586102594, 0.31775098316557593, 0.3193757209219385, 0.321517157646717, 0.32310564690252264, 0.325100326610767, 0.3269573210961271, 0.32867755427818035, 0.331334025929342, 0.33304253804553147, 0.33470914707146215, 0.336621549411116, 0.33927235945430684, 0.3406648147137322, 0.342